# 02 · Extract and cache representations

**Runtime:** T4 GPU. **Time:** roughly 10–20 minutes for both models (most of it is writing to Drive).

For Pythia-160M-deduped (revision `step143000`) and GPT-2 Small, every option text
`Question: {q}\nAnswer: {option}` is run through the frozen model in fp16, then pooled:

| layer name | what it is |
|---|---|
| `L3`, `L6` | raw residual stream after block 3 / 6 (`hidden_states[3]`, `[6]`) |
| `L12` | last block output **after** the final LayerNorm (default "final layer") |
| `L12raw` | last block output **before** the final LayerNorm |

Poolings: `mean` (over non-padding tokens) and `last` (last non-padding token).
Saved as float32 `WORKSPACE/cache/<model>/<layer>_<pooling>/<split>.npy`, shape `(n_items, 4, d)`.

E0 checks done here: dev is re-extracted with identical settings and compared with the cache
(max |diff| must be < 1e-3); a batch-size-32 re-extraction is reported for information.

**Resumable:** if Colab disconnects, just run again. Finished models are skipped.

In [ ]:
# ---- Setup (edit the two lines below once) ------------------------------------------
GITHUB_REPO = "YOUR_GITHUB_USERNAME/YOUR_REPO_NAME"              # repo that holds this code
WORKSPACE   = "/content/drive/MyDrive/optical_lm_workspace"      # splits, cache, runs, reports
# -------------------------------------------------------------------------------------
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["OPTLM_WORKSPACE"] = WORKSPACE
os.environ["HF_HOME"] = "/content/hf_cache"          # downloads stay on the fast local disk

# Private repo: add a GitHub token as a Colab secret named GH_TOKEN (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
REPO_DIR = "/content/" + GITHUB_REPO.split("/")[-1]
cmd = ["git", "clone", "-q", url, REPO_DIR] if not os.path.isdir(REPO_DIR) else ["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    msg = res.stderr.replace(token, "***") if token else res.stderr
    raise RuntimeError("git failed (check GITHUB_REPO / GH_TOKEN):\n" + msg)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("repo   :", REPO_DIR)
print("commit :", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip())
print("dirty  :", bool(subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"]).strip()))
print("workspace:", WORKSPACE)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!python scripts/extract_features.py --config configs/extract.yaml

### Cache summary

In [ ]:
import json, glob, os
for path in sorted(glob.glob(os.path.join(WORKSPACE, "cache", "*", "manifest.json"))):
    m = json.load(open(path))
    print(f"{m['model_tag']}: complete={m['complete']}, files={len(m['files'])}, d={m['hidden_size']}, "
          f"total {m.get('seconds_total')}s")
    for k, v in m.get("determinism", {}).get("results", {}).items():
        print(f"   {k:14s} same-settings diff {v['max_abs_diff_same_settings']:.1e}   "
              f"alt-batch diff {v['max_abs_diff_alt_batch_size']:.1e}   mean|h| {v['mean_abs_value']:.3f}")